# Chapter 6.3: Torch.compile Integration in SGLang

## Learning Objectives

By the end of this notebook, you will:

1. **Understand torch.compile** and its role in inference optimization
2. **Compare graph capture vs eager mode** execution
3. **Learn how SGLang integrates torch.compile** into its model execution pipeline
4. **Understand compilation warmup and caching** strategies
5. **Handle dynamic shapes** in compiled models
6. **Measure performance gains** from compilation
7. **Understand interaction** between torch.compile and CUDA graphs
8. **Debug compilation issues** in practice

## Prerequisites

- PyTorch 2.0+ fundamentals
- Understanding of computation graphs
- Basic knowledge of GPU kernel execution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part6/chapter_6.3_torch_compile.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part6/chapter_6.3_torch_compile.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Torch.compile Overview

### 1.1 What is torch.compile?

`torch.compile` (introduced in PyTorch 2.0) is a just-in-time (JIT) compiler that:

1. **Traces** the Python model code into an intermediate representation (FX graph)
2. **Optimizes** the graph (fusion, constant folding, dead code elimination)
3. **Generates** efficient GPU kernels via backends (Inductor, Triton)

```
Python Model Code
       |
       v
+------------------+
| TorchDynamo      |  <-- Python bytecode analysis & tracing
| (Graph Capture)  |      Identifies PyTorch operations
+------------------+
       |
       v
+------------------+
| AOTAutograd      |  <-- Ahead-of-time gradient computation
| (Graph Lowering) |      Decomposes ops to primitives
+------------------+
       |
       v
+------------------+
| Inductor Backend |  <-- Generates Triton/C++ kernels
| (Code Generation)|      Applies kernel fusion
+------------------+
       |
       v
  Optimized CUDA/Triton Kernels
```

### 1.2 Why torch.compile for LLM Inference?

LLM inference has several characteristics that make compilation beneficial:

| Characteristic | Benefit from Compilation |
|---------------|------------------------|
| Many small operations | Kernel fusion reduces launch overhead |
| Repeated structure (layers) | Compile once, reuse across layers |
| Predictable compute pattern | Graph optimization effective |
| Memory-bound operations | Fusion reduces memory traffic |

In [ ]:
import torch
import time

# Check PyTorch version and compilation support
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"torch.compile available: {hasattr(torch, 'compile')}")

In [ ]:
# Simple example: torch.compile on a basic operation

def simple_mlp(x, w1, w2, bias1, bias2):
    """A simple 2-layer MLP (eager mode)."""
    h = torch.nn.functional.silu(x @ w1.T + bias1)  # Layer 1 + SiLU activation
    return h @ w2.T + bias2  # Layer 2

# Create inputs
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 32
hidden_dim = 4096
inter_dim = 11008

x = torch.randn(batch_size, hidden_dim, device=device)
w1 = torch.randn(inter_dim, hidden_dim, device=device)
w2 = torch.randn(hidden_dim, inter_dim, device=device)
bias1 = torch.randn(inter_dim, device=device)
bias2 = torch.randn(hidden_dim, device=device)

# Compile the function
if hasattr(torch, 'compile'):
    compiled_mlp = torch.compile(simple_mlp, mode="reduce-overhead")
    print("Function compiled successfully.")
    print("\nCompilation modes available:")
    print("  'default'         - balanced optimization")
    print("  'reduce-overhead' - minimize CPU overhead (best for inference)")
    print("  'max-autotune'    - try all options, pick fastest")
else:
    compiled_mlp = simple_mlp
    print("torch.compile not available, using eager mode.")

In [ ]:
# Benchmark: Eager vs Compiled

def benchmark(fn, args, n_warmup=10, n_iter=100, name=""):
    """Benchmark a function with warmup."""
    # Warmup
    for _ in range(n_warmup):
        _ = fn(*args)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    start = time.perf_counter()
    for _ in range(n_iter):
        _ = fn(*args)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    elapsed = (time.perf_counter() - start) / n_iter * 1000  # ms
    print(f"{name:30s}: {elapsed:.3f} ms")
    return elapsed

args = (x, w1, w2, bias1, bias2)

print("=== Eager vs Compiled Performance ===")
print(f"Input shape: ({batch_size}, {hidden_dim})")
print(f"Device: {device}\n")

eager_time = benchmark(simple_mlp, args, name="Eager Mode")

if hasattr(torch, 'compile'):
    # First call triggers compilation
    print("\n(First compiled call includes compilation time)")
    compile_start = time.perf_counter()
    _ = compiled_mlp(*args)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    compile_time = (time.perf_counter() - compile_start) * 1000
    print(f"First call (with compilation): {compile_time:.1f} ms")
    
    compiled_time = benchmark(compiled_mlp, args, name="Compiled Mode")
    print(f"\nSpeedup: {eager_time / compiled_time:.2f}x")

## 2. Graph Capture vs Eager Mode

### 2.1 Eager Mode (Default PyTorch)

In eager mode, each PyTorch operation is dispatched immediately:

```python
# Each line launches a separate CUDA kernel
h = x @ w1.T       # Kernel 1: GEMM
h = h + bias1       # Kernel 2: element-wise add
h = F.silu(h)       # Kernel 3: SiLU activation
h = h @ w2.T        # Kernel 4: GEMM
h = h + bias2       # Kernel 5: element-wise add
```

**Problem:** 5 kernel launches, 5 memory round-trips. For small operations, the kernel launch overhead dominates.

### 2.2 Graph Capture Mode

With graph capture (torch.compile or CUDA graphs), operations are fused:

```python
# After compilation/fusion:
# Kernel 1: GEMM (x @ w1.T)          -- cannot fuse with non-GEMM
# Kernel 2: bias1 + SiLU (FUSED!)    -- element-wise ops fused together
# Kernel 3: GEMM (h @ w2.T)
# Kernel 4: bias2 add                -- or fused with GEMM epilogue
```

**Benefits:**
- Fewer kernel launches (5 -> 3 or fewer)
- Reduced memory traffic (intermediate values stay in registers)
- Eliminated Python overhead between operations

### 2.3 Comparison

```
                Eager Mode                    Graph Capture
            +-----------------+           +-----------------+
  Python -> | op1 | op2 | op3| Python ->  | fused_op(1,2,3)|
  overhead  | ^   | ^   | ^  | overhead   | ^              |
            | |   | |   | |  |            | |              |
            +-v---+-v---+-v--+             +-v--------------+
  GPU       |K1  |K2  |K3   |  GPU       |K_fused          |
            |    |    |      |            |                 |
            +----+----+------+            +-----------------+
              Time: 3x launch               Time: 1x launch
              + 3x memory access            + 1x memory access
```

In [ ]:
# Demonstrate kernel fusion with torch.compile

def unfused_operations(x):
    """Multiple element-wise operations (not fused)."""
    x = x * 2.0
    x = x + 1.0
    x = torch.relu(x)
    x = x * 0.5
    x = x - 0.3
    x = torch.tanh(x)
    return x

# In eager mode: 6 separate kernels
# After compilation: 1 fused kernel!

x_test = torch.randn(1024, 4096, device=device)

print("=== Kernel Fusion Demo ===")
print("6 element-wise operations on a (1024, 4096) tensor\n")

eager_time = benchmark(unfused_operations, (x_test,), name="Eager (6 kernels)")

if hasattr(torch, 'compile'):
    compiled_ops = torch.compile(unfused_operations, mode="reduce-overhead")
    # Warmup compilation
    _ = compiled_ops(x_test)
    compiled_time = benchmark(compiled_ops, (x_test,), name="Compiled (1 fused kernel)")
    print(f"\nSpeedup from fusion: {eager_time / compiled_time:.2f}x")
    print("Note: Fusion benefit is largest for element-wise operations.")

## 3. How SGLang Integrates torch.compile

### 3.1 Integration Points

SGLang applies torch.compile at the **model forward pass** level, not individual operations. The key integration points are:

```python
# From sglang/srt/model_executor/model_runner.py

class ModelRunner:
    def __init__(self, model_config, ...):
        self.model = self.load_model()
        
        # Apply torch.compile if enabled
        if server_args.enable_torch_compile:
            self.model = torch.compile(
                self.model,
                mode="reduce-overhead",  # Best for inference
                fullgraph=False,          # Allow graph breaks
                dynamic=True,             # Support dynamic shapes
            )
    
    def forward_batch(self, batch):
        """Execute the model on a batch."""
        # torch.compile captures the graph on first call,
        # subsequent calls with same shape reuse compiled graph
        output = self.model(
            input_ids=batch.input_ids,
            positions=batch.positions,
            ...
        )
        return output
```

### 3.2 Compilation Strategy

SGLang uses several strategies to make torch.compile effective:

1. **`reduce-overhead` mode**: Minimizes Python overhead (uses CUDA graphs internally)
2. **`dynamic=True`**: Handles varying batch sizes and sequence lengths
3. **`fullgraph=False`**: Allows graph breaks for unsupported operations
4. **Selective compilation**: Only compile the main forward pass, not sampling or post-processing

### 3.3 What Gets Compiled

| Component | Compiled? | Why |
|-----------|-----------|-----|
| Attention layers | Yes | Significant fusion opportunity |
| MLP/FFN layers | Yes | Large compute, fusible activations |
| RMSNorm | Yes | Element-wise, excellent fusion target |
| Rotary embeddings | Yes | Fuses with attention |
| Sampling (top-k/p) | No | Dynamic control flow |
| Token tree verification | No | Complex branching logic |

In [ ]:
# Simulate SGLang's compilation approach with a transformer layer

class RMSNorm(torch.nn.Module):
    """Root Mean Square Layer Normalization."""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = torch.nn.Parameter(torch.ones(dim))
        self.eps = eps
    
    def forward(self, x):
        norm = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * norm * self.weight

class SimpleMLP(torch.nn.Module):
    """Simplified LLM MLP block (SwiGLU)."""
    def __init__(self, hidden_dim, inter_dim):
        super().__init__()
        self.gate_proj = torch.nn.Linear(hidden_dim, inter_dim, bias=False)
        self.up_proj = torch.nn.Linear(hidden_dim, inter_dim, bias=False)
        self.down_proj = torch.nn.Linear(inter_dim, hidden_dim, bias=False)
    
    def forward(self, x):
        # SwiGLU activation: silu(gate(x)) * up(x)
        gate = torch.nn.functional.silu(self.gate_proj(x))
        up = self.up_proj(x)
        return self.down_proj(gate * up)

class SimpleTransformerLayer(torch.nn.Module):
    """Simplified transformer layer for compilation demo."""
    def __init__(self, hidden_dim=512, n_heads=8, inter_dim=1536):
        super().__init__()
        self.attn_norm = RMSNorm(hidden_dim)
        self.mlp_norm = RMSNorm(hidden_dim)
        
        # Simplified attention (no actual attention computation)
        self.qkv_proj = torch.nn.Linear(hidden_dim, 3 * hidden_dim, bias=False)
        self.o_proj = torch.nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.n_heads = n_heads
        self.head_dim = hidden_dim // n_heads
        
        self.mlp = SimpleMLP(hidden_dim, inter_dim)
    
    def forward(self, x):
        # Attention block with residual
        h = self.attn_norm(x)
        qkv = self.qkv_proj(h)
        q, k, v = qkv.chunk(3, dim=-1)
        # Simplified: just project through o_proj (skip actual attention)
        attn_out = self.o_proj(v)  # Placeholder for real attention
        h = x + attn_out
        
        # MLP block with residual
        mlp_out = self.mlp(self.mlp_norm(h))
        return h + mlp_out

# Create model
hidden_dim = 512
layer = SimpleTransformerLayer(hidden_dim=hidden_dim).to(device).eval()

print(f"Model created on {device}")
print(f"Parameters: {sum(p.numel() for p in layer.parameters()):,}")

In [ ]:
# Compile and benchmark the transformer layer

x_input = torch.randn(32, 128, hidden_dim, device=device)

print("=== Transformer Layer: Eager vs Compiled ===")
print(f"Input shape: {x_input.shape}\n")

with torch.no_grad():
    eager_time = benchmark(layer, (x_input,), name="Eager Mode")
    
    if hasattr(torch, 'compile'):
        compiled_layer = torch.compile(layer, mode="reduce-overhead")
        
        # Compilation warmup
        print("\nCompiling...")
        compile_start = time.perf_counter()
        _ = compiled_layer(x_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        print(f"Compilation time: {(time.perf_counter() - compile_start)*1000:.0f} ms")
        
        compiled_time = benchmark(compiled_layer, (x_input,), name="\nCompiled Mode")
        print(f"\nSpeedup: {eager_time / compiled_time:.2f}x")

## 4. Compilation Warmup and Caching

### 4.1 The Warmup Problem

torch.compile has a **cold start penalty**: the first call triggers compilation which can take seconds to minutes.

```
Timeline without warmup:
  t=0: Server starts
  t=1: First request arrives
  t=1-15s: COMPILATION (request blocked!)
  t=15s: First response returned (very slow)
  t=15s+: Subsequent requests fast (compiled)

Timeline with warmup:
  t=0: Server starts
  t=0-15s: WARMUP COMPILATION (no requests yet)
  t=15s: Server ready
  t=15s+: First request arrives, immediately fast
```

### 4.2 SGLang Warmup Strategy

```python
# From sglang/srt/model_executor/model_runner.py

def warmup_compile(self):
    """
    Run warmup passes with representative input shapes
    to trigger compilation before serving requests.
    """
    warmup_shapes = [
        # (batch_size, seq_len) tuples covering common cases
        (1, 1),      # Single decode token
        (1, 128),    # Short prefill
        (1, 512),    # Medium prefill
        (1, 2048),   # Long prefill
        (32, 1),     # Batch decode
        (64, 1),     # Large batch decode
    ]
    
    for batch_size, seq_len in warmup_shapes:
        dummy_input = self.create_dummy_input(batch_size, seq_len)
        with torch.no_grad():
            _ = self.model(dummy_input)
    
    print(f"Warmup complete: compiled {len(warmup_shapes)} shapes")
```

### 4.3 Compilation Cache

PyTorch caches compiled kernels to disk, avoiding recompilation on server restart:

```python
import torch._inductor.config

# Enable persistent compilation cache
torch._inductor.config.fx_graph_cache = True
torch._inductor.config.fx_graph_remote_cache = False  # Local cache only

# Cache location: ~/.cache/torch/inductor/
```

In [ ]:
# Demonstrate compilation caching and multiple shapes

if hasattr(torch, 'compile'):
    # Create a fresh compiled model
    layer_v2 = SimpleTransformerLayer(hidden_dim=512).to(device).eval()
    compiled_v2 = torch.compile(layer_v2, mode="reduce-overhead", dynamic=True)
    
    shapes = [
        (1, 1, 512),     # Single token decode
        (1, 128, 512),   # Short prefill
        (32, 1, 512),    # Batch decode
        (1, 512, 512),   # Medium prefill
    ]
    
    print("=== Compilation for Multiple Shapes ===")
    print("(First call for each shape triggers compilation)\n")
    
    with torch.no_grad():
        for shape in shapes:
            x = torch.randn(*shape, device=device)
            
            # First call: compilation
            start = time.perf_counter()
            _ = compiled_v2(x)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            first_time = (time.perf_counter() - start) * 1000
            
            # Second call: cached
            start = time.perf_counter()
            _ = compiled_v2(x)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            cached_time = (time.perf_counter() - start) * 1000
            
            print(f"Shape {str(shape):20s}: First={first_time:8.1f}ms, Cached={cached_time:6.3f}ms")
else:
    print("torch.compile not available.")

## 5. Dynamic Shapes Handling

### 5.1 The Dynamic Shape Challenge

LLM inference has **inherently dynamic shapes**:
- Batch size varies (different number of concurrent requests)
- Sequence length varies (prefill vs decode, different prompt lengths)
- KV-cache size grows during generation

Without dynamic shape support, torch.compile would recompile for every new shape combination, making it impractical.

### 5.2 Solutions

**1. `dynamic=True` flag:**
```python
compiled_model = torch.compile(model, dynamic=True)
# Generates generic kernels that handle varying sizes
# Trade-off: slightly less optimal than static compilation
```

**2. Padding to fixed shapes:**
```python
# Pad batch to power-of-2 sizes
VALID_BATCH_SIZES = [1, 2, 4, 8, 16, 32, 64, 128]

def pad_to_valid_size(batch_size):
    for valid in VALID_BATCH_SIZES:
        if batch_size <= valid:
            return valid
    return batch_size
```

**3. Bucketing:**
```python
# Pre-compile for common shapes
compiled_models = {}
for bs in [1, 4, 16, 64]:
    for seq in [1, 128, 512, 2048]:
        compiled_models[(bs, seq)] = compile_for_shape(model, bs, seq)
```

### 5.3 SGLang's Approach

SGLang uses a combination:
- `dynamic=True` for flexibility
- Padding decode batches to power-of-2 sizes
- Separate compilation for prefill and decode phases

In [ ]:
# Dynamic shapes example

def demonstrate_dynamic_shapes():
    """Show how dynamic=True handles varying input sizes."""
    if not hasattr(torch, 'compile'):
        print("torch.compile not available.")
        return
    
    model = SimpleMLP(hidden_dim=256, inter_dim=768).to(device).eval()
    
    # Compile with dynamic=True
    compiled_dynamic = torch.compile(model, dynamic=True)
    # Compile with dynamic=False (static)
    compiled_static = torch.compile(model, dynamic=False)
    
    test_shapes = [(1, 256), (4, 256), (16, 256), (32, 256), (7, 256), (13, 256)]
    
    print("=== Dynamic vs Static Shape Handling ===")
    print(f"{'Shape':>12s} | {'Dynamic':>12s} | {'Static':>12s} | {'Note':>20s}")
    print("-" * 65)
    
    with torch.no_grad():
        for shape in test_shapes:
            x = torch.randn(*shape, device=device)
            
            # Dynamic: should work for all shapes
            try:
                start = time.perf_counter()
                _ = compiled_dynamic(x)
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                dynamic_time = (time.perf_counter() - start) * 1000
                dynamic_result = f"{dynamic_time:.1f}ms"
            except Exception as e:
                dynamic_result = "Error"
            
            # Static: may recompile for each new shape
            try:
                start = time.perf_counter()
                _ = compiled_static(x)
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                static_time = (time.perf_counter() - start) * 1000
                static_result = f"{static_time:.1f}ms"
                note = "recompiled" if static_time > 100 else "cached"
            except Exception as e:
                static_result = "Error"
                note = str(e)[:20]
            
            print(f"{str(shape):>12s} | {dynamic_result:>12s} | {static_result:>12s} | {note:>20s}")

demonstrate_dynamic_shapes()

## 6. Performance Gains from Compilation

### 6.1 Where Does the Speedup Come From?

```
+--------------------------------------------------+
|              Compilation Speedup Sources          |
+--------------------------------------------------+
| 1. Kernel Fusion                                  |
|    - Multiple element-wise ops -> 1 kernel        |
|    - Reduces memory traffic and launch overhead   |
|    - Typical: 1.5-3x for element-wise chains     |
+--------------------------------------------------+
| 2. Python Overhead Elimination                    |
|    - No Python interpreter between ops            |
|    - Significant for small operations             |
|    - Typical: 1.1-1.3x                           |
+--------------------------------------------------+
| 3. Memory Layout Optimization                     |
|    - Better data placement for GPU access         |
|    - Stride optimization                          |
|    - Typical: 1.05-1.2x                          |
+--------------------------------------------------+
| 4. Operator Selection                             |
|    - Auto-tune best kernel implementation         |
|    - Platform-specific optimizations              |
|    - Typical: 1.0-1.3x                           |
+--------------------------------------------------+

Total typical speedup: 1.2-2.5x for LLM inference
```

### 6.2 Benchmark Results (Representative)

| Model | Eager (tok/s) | Compiled (tok/s) | Speedup |
|-------|--------------|-----------------|--------|
| Llama-3-8B (A100) | 85 | 105 | 1.24x |
| Llama-3-70B (8xA100) | 42 | 52 | 1.24x |
| Mistral-7B (A100) | 95 | 118 | 1.24x |

*Note: Actual numbers vary by hardware, batch size, and sequence length.*

In [ ]:
# Comprehensive benchmark: multiple model sizes

class MultiLayerModel(torch.nn.Module):
    """Multi-layer transformer for benchmarking."""
    def __init__(self, hidden_dim, n_layers, inter_dim=None):
        super().__init__()
        inter_dim = inter_dim or hidden_dim * 4
        self.layers = torch.nn.ModuleList([
            SimpleTransformerLayer(hidden_dim=hidden_dim, 
                                  n_heads=max(hidden_dim//64, 1),
                                  inter_dim=inter_dim)
            for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(hidden_dim)
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.final_norm(x)

configs = [
    {"name": "Small (128d, 4L)", "hidden": 128, "layers": 4},
    {"name": "Medium (256d, 8L)", "hidden": 256, "layers": 8},
    {"name": "Large (512d, 12L)", "hidden": 512, "layers": 12},
]

batch_size = 16
seq_len = 64

print("=== Multi-Size Compilation Benchmark ===")
print(f"Batch: {batch_size}, Seq: {seq_len}\n")
print(f"{'Config':>25s} | {'Params':>10s} | {'Eager(ms)':>10s} | {'Compiled(ms)':>12s} | {'Speedup':>8s}")
print("-" * 75)

for cfg in configs:
    model = MultiLayerModel(cfg["hidden"], cfg["layers"]).to(device).eval()
    n_params = sum(p.numel() for p in model.parameters())
    x = torch.randn(batch_size, seq_len, cfg["hidden"], device=device)
    
    with torch.no_grad():
        eager_t = benchmark(model, (x,), n_warmup=5, n_iter=50, name="")
        
        if hasattr(torch, 'compile'):
            compiled = torch.compile(model, mode="reduce-overhead")
            _ = compiled(x)  # Warmup
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            compiled_t = benchmark(compiled, (x,), n_warmup=5, n_iter=50, name="")
            speedup = eager_t / compiled_t
        else:
            compiled_t = eager_t
            speedup = 1.0
    
    print(f"{cfg['name']:>25s} | {n_params:>10,} | {eager_t:>9.3f} | {compiled_t:>11.3f} | {speedup:>7.2f}x")

## 7. Interaction with CUDA Graphs

### 7.1 CUDA Graphs Overview

CUDA graphs capture a sequence of GPU operations and replay them as a single unit:

```python
# Capture phase
with torch.cuda.graph(graph):
    output = model(input_tensor)

# Replay phase (very fast)
input_tensor.copy_(new_input)
graph.replay()  # Replays all captured operations
```

### 7.2 torch.compile + CUDA Graphs

When using `mode="reduce-overhead"`, torch.compile **internally uses CUDA graphs**:

```
torch.compile(mode="reduce-overhead")
  |
  +--> TorchDynamo traces the graph
  +--> Inductor generates optimized kernels
  +--> CUDAGraphModule wraps kernels in CUDA graph
  +--> First call: capture CUDA graph
  +--> Subsequent calls: replay CUDA graph
```

### 7.3 SGLang's CUDA Graph Strategy

SGLang also has its own CUDA graph implementation (independent of torch.compile):

```python
# From sglang/srt/model_executor/cuda_graph_runner.py

class CudaGraphRunner:
    """
    Manages CUDA graphs for the decode phase.
    Pre-captures graphs for common batch sizes.
    """
    def __init__(self, model_runner):
        self.graphs = {}  # batch_size -> captured graph
        
        # Pre-capture for common batch sizes
        for bs in [1, 2, 4, 8, 16, 32, 64, 128]:
            self.graphs[bs] = self._capture_graph(model_runner, bs)
    
    def _capture_graph(self, model_runner, batch_size):
        # Create static input tensors
        static_input_ids = torch.zeros(batch_size, 1, dtype=torch.long, device='cuda')
        # ... other static tensors
        
        # Warmup run (required before capture)
        model_runner.forward(static_input_ids, ...)
        
        # Capture
        graph = torch.cuda.CUDAGraph()
        with torch.cuda.graph(graph):
            output = model_runner.forward(static_input_ids, ...)
        
        return graph, static_input_ids, output
    
    def run(self, input_ids, ...):
        batch_size = input_ids.shape[0]
        # Find matching graph (pad if needed)
        padded_bs = self._pad_to_valid(batch_size)
        graph, static_input, static_output = self.graphs[padded_bs]
        
        # Copy actual data into static buffers
        static_input[:batch_size].copy_(input_ids)
        
        # Replay graph
        graph.replay()
        
        return static_output[:batch_size]
```

### 7.4 When to Use Which?

| Feature | torch.compile | CUDA Graphs (manual) | torch.compile + reduce-overhead |
|---------|--------------|---------------------|-------------------------------|
| Kernel fusion | Yes | No | Yes |
| Launch overhead | Reduced | Minimal | Minimal |
| Dynamic shapes | Yes (with `dynamic=True`) | No | Limited |
| Setup complexity | Low | High | Low |
| Best for | Prefill phase | Decode phase | Both |

In [ ]:
# Demonstrate CUDA Graph capture and replay

if torch.cuda.is_available():
    model_cg = SimpleMLP(256, 768).cuda().eval()
    
    # Static input/output buffers
    static_input = torch.randn(32, 256, device='cuda')
    
    # Warmup
    with torch.no_grad():
        for _ in range(3):
            _ = model_cg(static_input)
    
    # Capture CUDA graph
    graph = torch.cuda.CUDAGraph()
    with torch.cuda.graph(graph):
        static_output = model_cg(static_input)
    
    print("CUDA Graph captured!")
    print(f"Input shape: {static_input.shape}")
    print(f"Output shape: {static_output.shape}")
    
    # Benchmark: Eager vs CUDA Graph vs torch.compile
    print("\n=== Three-Way Comparison ===")
    
    with torch.no_grad():
        # Eager
        eager_t = benchmark(model_cg, (static_input,), name="Eager")
        
        # CUDA Graph replay
        def graph_replay():
            graph.replay()
            return static_output
        
        cuda_graph_t = benchmark(lambda: graph_replay(), (), name="CUDA Graph")
        
        # torch.compile
        if hasattr(torch, 'compile'):
            compiled = torch.compile(model_cg, mode="reduce-overhead")
            _ = compiled(static_input)  # warmup
            torch.cuda.synchronize()
            compile_t = benchmark(compiled, (static_input,), name="torch.compile")
            
            print(f"\nCUDA Graph speedup over eager: {eager_t/cuda_graph_t:.2f}x")
            print(f"torch.compile speedup over eager: {eager_t/compile_t:.2f}x")
else:
    print("CUDA not available. CUDA graphs require a GPU.")
    print("\nConceptual explanation:")
    print("  CUDA graphs eliminate Python/CPU overhead entirely.")
    print("  The entire forward pass replays as a single GPU operation.")
    print("  Best for the decode phase where shapes are fixed.")

## 8. Debugging Compilation Issues

### 8.1 Common Issues

| Issue | Symptom | Solution |
|-------|---------|--------|
| Graph breaks | Warning messages, no speedup | Refactor code to avoid breaks |
| Dynamic control flow | Compilation fails | Use `dynamic=True` or pad inputs |
| Unsupported ops | RuntimeError | Use `fullgraph=False` or rewrite op |
| Memory explosion | OOM during compilation | Reduce model size or use `backend="eager"` |
| Numerical mismatch | Different outputs | Check precision settings |

### 8.2 Graph Breaks

A **graph break** occurs when torch.compile encounters an operation it cannot trace. The graph is split into segments:

```python
def model_with_break(x):
    h = torch.relu(x)          # Segment 1 (compiled)
    print(h.shape)              # GRAPH BREAK! (print is not traceable)
    h = torch.sigmoid(h)        # Segment 2 (compiled separately)
    return h
```

Graph breaks prevent fusion across segments, reducing optimization benefit.

### 8.3 Debugging Tools

```python
# Enable verbose compilation logging
import torch._dynamo
torch._dynamo.config.verbose = True

# Count graph breaks
torch._dynamo.config.log_graph_breaks = True

# Explain why compilation fails
torch._dynamo.explain(model, input)
```

In [ ]:
# Demonstrate graph breaks and how to fix them

class ModelWithBreaks(torch.nn.Module):
    """Model that causes graph breaks."""
    def __init__(self, dim):
        super().__init__()
        self.linear1 = torch.nn.Linear(dim, dim)
        self.linear2 = torch.nn.Linear(dim, dim)
    
    def forward(self, x):
        h = torch.relu(self.linear1(x))
        
        # GRAPH BREAK: data-dependent control flow
        if h.mean() > 0:  # This check depends on runtime values!
            h = h * 2
        else:
            h = h * 3
        
        return self.linear2(h)

class ModelWithoutBreaks(torch.nn.Module):
    """Fixed model that avoids graph breaks."""
    def __init__(self, dim):
        super().__init__()
        self.linear1 = torch.nn.Linear(dim, dim)
        self.linear2 = torch.nn.Linear(dim, dim)
    
    def forward(self, x):
        h = torch.relu(self.linear1(x))
        
        # FIXED: Use torch.where instead of if/else
        mask = (h.mean() > 0).float()
        h = h * (2 * mask + 3 * (1 - mask))
        
        return self.linear2(h)

print("=== Graph Break Analysis ===")
print("\nModel with graph breaks:")
print("  if h.mean() > 0: ...  <-- Data-dependent branch causes break")
print("\nFixed model:")
print("  torch.where or masking  <-- No break, fully compilable")

if hasattr(torch, 'compile') and hasattr(torch, '_dynamo'):
    model_breaks = ModelWithBreaks(256).to(device).eval()
    model_fixed = ModelWithoutBreaks(256).to(device).eval()
    x = torch.randn(16, 256, device=device)
    
    try:
        explanation = torch._dynamo.explain(model_breaks, x)
        print(f"\nGraph break analysis: {explanation}")
    except Exception as e:
        print(f"\nDynamo explain: {type(e).__name__}")
        print("(Graph break detection may vary by PyTorch version)")

In [ ]:
# Common patterns that cause graph breaks and their fixes

print("=== Common Graph Break Patterns and Fixes ===")

patterns = [
    {
        "name": "Data-dependent branching",
        "bad": """if x.sum() > 0:
    x = x * 2
else:
    x = x * 3""",
        "good": """mask = (x.sum() > 0).float()
x = x * (2 * mask + 3 * (1 - mask))""",
    },
    {
        "name": "Python print/logging",
        "bad": """h = model(x)
print(f'Output shape: {h.shape}')  # BREAK!
return h""",
        "good": """h = model(x)
# Move logging outside compiled function
return h""",
    },
    {
        "name": "Python list operations on tensors",
        "bad": """outputs = []
for layer in self.layers:
    x = layer(x)
    outputs.append(x)  # BREAK: Python list""",
        "good": """for layer in self.layers:
    x = layer(x)
# Or use torch.stack if needed""",
    },
    {
        "name": "Tensor.item() calls",
        "bad": """loss_val = loss.item()  # BREAK: Python scalar conversion
if loss_val > threshold: ...""",
        "good": """# Keep everything as tensors
mask = (loss > threshold).float()""",
    },
]

for p in patterns:
    print(f"\n--- {p['name']} ---")
    print(f"  BAD (causes break):")
    for line in p['bad'].split('\n'):
        print(f"    {line}")
    print(f"  GOOD (no break):")
    for line in p['good'].split('\n'):
        print(f"    {line}")

## 9. Source Code Walkthrough: SGLang Integration

### 9.1 Enabling torch.compile

```python
# Launch command:
# python -m sglang.launch_server --model meta-llama/Llama-3-8B \
#     --enable-torch-compile

# In server_args.py:
class ServerArgs:
    enable_torch_compile: bool = False
    torch_compile_max_bs: int = 32  # Max batch size for compilation
```

### 9.2 Model Runner Integration

```python
# In model_runner.py:

class ModelRunner:
    def init_torch_compile(self):
        """Apply torch.compile to the model."""
        if not self.server_args.enable_torch_compile:
            return
        
        # Set compilation config
        torch._inductor.config.coordinate_descent_tuning = True
        torch._inductor.config.triton.unique_kernel_names = True
        torch._inductor.config.fx_graph_cache = True  # Persistent cache
        
        # Compile the model
        self.model.forward = torch.compile(
            self.model.forward,
            mode="max-autotune-no-cudagraphs",  # Let SGLang manage CUDA graphs
            dynamic=False,  # SGLang pads to fixed shapes
        )
        
        # Warmup with representative shapes
        self._warmup_compile()
```

### 9.3 Key Design Decision: No CUDA Graphs in torch.compile

SGLang uses `mode="max-autotune-no-cudagraphs"` because it manages CUDA graphs separately through `CudaGraphRunner`. This avoids conflicts between:
- torch.compile's internal CUDA graph management
- SGLang's explicit CUDA graph capture for decode batches

```
SGLang's approach:
  torch.compile: kernel fusion + code generation (no CUDA graphs)
  CudaGraphRunner: CUDA graph capture for decode phase
  
  Result: Best of both worlds
    - Fused kernels from torch.compile
    - Zero-overhead replay from CUDA graphs
```

In [ ]:
# Simulate SGLang's torch.compile integration

class SGLangModelRunner:
    """Simplified SGLang model runner with torch.compile support."""
    
    def __init__(self, model, enable_compile=False, enable_cuda_graphs=False):
        self.model = model
        self.enable_compile = enable_compile
        self.enable_cuda_graphs = enable_cuda_graphs
        self.cuda_graphs = {}  # batch_size -> (graph, static_input, static_output)
        
        if enable_compile and hasattr(torch, 'compile'):
            print("Applying torch.compile...")
            self.model = torch.compile(
                self.model,
                mode="max-autotune" if not enable_cuda_graphs else "default",
            )
        
        if enable_cuda_graphs and torch.cuda.is_available():
            self._capture_cuda_graphs()
    
    def _capture_cuda_graphs(self):
        """Pre-capture CUDA graphs for common batch sizes."""
        for bs in [1, 4, 16, 32]:
            try:
                static_in = torch.randn(bs, 64, 256, device='cuda')
                # Warmup
                with torch.no_grad():
                    for _ in range(3):
                        _ = self.model(static_in)
                
                graph = torch.cuda.CUDAGraph()
                with torch.cuda.graph(graph):
                    static_out = self.model(static_in)
                
                self.cuda_graphs[bs] = (graph, static_in, static_out)
                print(f"  Captured CUDA graph for batch_size={bs}")
            except Exception as e:
                print(f"  Failed to capture graph for bs={bs}: {e}")
    
    def forward(self, x):
        bs = x.shape[0]
        
        if self.enable_cuda_graphs and bs in self.cuda_graphs:
            graph, static_in, static_out = self.cuda_graphs[bs]
            static_in.copy_(x)
            graph.replay()
            return static_out.clone()
        
        return self.model(x)

# Create and test
base_model = MultiLayerModel(hidden_dim=256, n_layers=4).to(device).eval()
print("=== SGLang Model Runner Modes ===")

runners = {}
with torch.no_grad():
    print("\n1. Eager mode:")
    runners["eager"] = SGLangModelRunner(base_model, enable_compile=False)
    
    if hasattr(torch, 'compile'):
        print("\n2. torch.compile mode:")
        model_copy = MultiLayerModel(hidden_dim=256, n_layers=4).to(device).eval()
        runners["compile"] = SGLangModelRunner(model_copy, enable_compile=True)

print("\nRunners created successfully.")

## 10. Summary

### Key Takeaways

1. **torch.compile** traces Python model code into optimized GPU kernels via TorchDynamo + Inductor backend.

2. **Kernel fusion** is the primary speedup source -- combining multiple operations into single GPU kernels reduces memory traffic and launch overhead.

3. **SGLang integrates torch.compile** selectively: compile the model forward pass, but manage CUDA graphs separately.

4. **Compilation warmup** is essential to avoid cold-start latency spikes. SGLang pre-compiles for common input shapes.

5. **Dynamic shapes** are handled via `dynamic=True` or input padding to fixed sizes.

6. **Graph breaks** must be avoided for maximum optimization -- avoid Python control flow, print statements, and `.item()` calls in compiled code.

7. **CUDA graphs** complement torch.compile: compile provides fusion, CUDA graphs provide zero-overhead replay.

8. **Typical speedup** from torch.compile in LLM inference is **1.2-1.5x**, with additional gains from CUDA graphs.

## Exercises

### Exercise 1: Debug a Compilation Issue

The following model has graph breaks. Identify and fix them:

```python
class BuggyModel(torch.nn.Module):
    def forward(self, x):
        h = self.layer1(x)
        print(f"Layer 1 output mean: {h.mean().item()}")  # Bug 1
        
        if h.max().item() > 10:  # Bug 2
            h = h / 10
        
        outputs = []
        for i in range(h.shape[0]):  # Bug 3
            outputs.append(self.layer2(h[i:i+1]))
        
        return torch.cat(outputs)
```

### Exercise 2: Benchmark Compilation Modes

Compare all compilation modes for a 6-layer transformer:
- `mode="default"`
- `mode="reduce-overhead"` 
- `mode="max-autotune"`

Measure: compilation time, inference latency, memory usage.

### Exercise 3: Custom Compilation Strategy

Implement a custom compilation manager that:
1. Monitors input shape distribution
2. Pre-compiles for the top-K most common shapes
3. Falls back to eager mode for rare shapes
4. Tracks compilation cache hit rate

In [ ]:
# Exercise 1 Starter Code

class BuggyModel(torch.nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.layer1 = torch.nn.Linear(dim, dim)
        self.layer2 = torch.nn.Linear(dim, dim)
    
    def forward(self, x):
        h = self.layer1(x)
        print(f"Layer 1 output mean: {h.mean().item()}")  # Bug 1: print + .item()
        
        if h.max().item() > 10:  # Bug 2: data-dependent branch + .item()
            h = h / 10
        
        outputs = []
        for i in range(h.shape[0]):  # Bug 3: dynamic loop over batch dim
            outputs.append(self.layer2(h[i:i+1]))
        
        return torch.cat(outputs)

class FixedModel(torch.nn.Module):
    """TODO: Fix all graph breaks from BuggyModel."""
    def __init__(self, dim):
        super().__init__()
        self.layer1 = torch.nn.Linear(dim, dim)
        self.layer2 = torch.nn.Linear(dim, dim)
    
    def forward(self, x):
        # YOUR CODE HERE
        # Fix Bug 1: Remove print and .item()
        # Fix Bug 2: Use torch.where or torch.clamp
        # Fix Bug 3: Apply layer2 to full batch at once
        pass

print("Exercise 1: Fix the BuggyModel to eliminate graph breaks.")
print("Hints:")
print("  Bug 1: Remove print and .item() calls")
print("  Bug 2: Use torch.where(h.max() > 10, h/10, h)")
print("  Bug 3: Apply layer2 to the entire batch: self.layer2(h)")

In [ ]:
# Exercise 3 Starter Code: Custom Compilation Manager

class CompilationManager:
    """Manages compilation for varying input shapes."""
    
    def __init__(self, model, top_k_shapes=5):
        self.model = model
        self.compiled_models = {}  # shape -> compiled model
        self.shape_counts = {}     # shape -> access count
        self.top_k = top_k_shapes
        self.stats = {"compiled_hits": 0, "eager_fallbacks": 0}
    
    def forward(self, x):
        shape = tuple(x.shape)
        
        # Track shape frequency
        self.shape_counts[shape] = self.shape_counts.get(shape, 0) + 1
        
        # TODO: Implement the logic
        # 1. If shape has a compiled version, use it
        # 2. If shape is frequent enough, compile and cache
        # 3. Otherwise, fall back to eager
        pass
    
    def get_stats(self):
        total = self.stats["compiled_hits"] + self.stats["eager_fallbacks"]
        return {
            "compiled_shapes": len(self.compiled_models),
            "unique_shapes_seen": len(self.shape_counts),
            "cache_hit_rate": self.stats["compiled_hits"] / total if total > 0 else 0,
        }

print("Exercise 3: Implement the CompilationManager.forward() method.")